In [64]:
import nflreadpy as nfl
import polars as pl

games = nfl.load_schedules([2019, 2020, 2021, 2022, 2023, 2024, 2025])

# Drop unnecessary columns
games = games.drop(['game_id', 'gameday', 'weekday', 'gametime', 'location', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium'])

# Filter to only regular season games
games = games.filter(pl.col("game_type") == "REG")

# games.tail()

In [65]:
# Create a column to specify whether the home team won or lost
games = games.with_columns(
    (pl.col('result') > 0).cast(pl.Int8).alias('home_team_win')
)

# games.tail()

In [66]:
# Create dataframe with the home teams games and results
home = games.select(['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score', 'home_team_win'])
home.columns = ['season', 'week', 'team', 'opponent', 'points_scored', 'points_allowed', 'won']

# Create dataframe with the away teams games and results
away = games.select(['season', 'week', 'away_team', 'home_team', 'away_score', 'home_score', 'home_team_win'])
away = away.with_columns(
    (1 - pl.col('home_team_win')).alias('home_team_win')
)
away.columns = ['season', 'week', 'team', 'opponent', 'points_scored', 'points_allowed', 'won']

# Stack the dataframes
games_full = pl.concat([home, away], how='vertical')

# games_melted.head(10)

In [67]:
games_full = games_full.sort(
    ['season', 'week']
)

# The points per game/points allowed per game are the points scored/allowed from previous games going into the current game
games_full = games_full.with_columns(
    # Using the won column, get the cumulative sum (cum_sum()), 
    # shifted back by 1 (shift(1)) to exclude the current game, 
    # then over each season and week (over(['season', 'week']))
    (pl.col('won').cum_sum().over(['team', 'season']) / 
     # same as above but using the count of the won column (which would be total games played)
     # over() partitions the data by the team and season, so the sum and count are calculated separately for each team and season
     pl.col('won').cum_count().over(['team', 'season'])).alias('win_percentage'),


    # Creates a rolling points per game average for each team and season
    (pl.col('points_scored').cum_sum().over(['team', 'season']) / 
     pl.col('points_scored').cum_count().over(['team', 'season'])).alias('points_per_game'),

    # Creates a rolling allowed points per game average for each team and season
    (pl.col('points_allowed').cum_sum().over(['team', 'season']) / 
     pl.col('points_allowed').cum_count().over(['team', 'season'])).alias('allowed_points_per_game')
).with_columns(
    # Shift all rows down by 1 to exclude the current game
    # We do this after creating the columns because shift() function will run before the window function if it's not separated like this
    pl.col('win_percentage').shift(1).over(['team', 'season']),
    pl.col('points_per_game').shift(1).over(['team', 'season']),
    pl.col('allowed_points_per_game').shift(1).over(['team', 'season'])
)

# games_full = games_full.filter(
#     (pl.col('team') == 'SEA') &
#     (pl.col('season') == 2024)
# )

games_full.tail()

season,week,team,opponent,points_scored,points_allowed,won,win_percentage,points_per_game,allowed_points_per_game
i32,i32,str,str,i32,i32,i8,f64,f64,f64
2025,18,"""DAL""","""NYG""",17,34,0,0.4375,28.375,29.8125
2025,18,"""WAS""","""PHI""",24,17,1,0.25,20.75,27.125
2025,18,"""BAL""","""PIT""",24,26,0,0.5,25.0,23.25
2025,18,"""SEA""","""SF""",13,3,1,0.8125,29.375,18.0625
2025,18,"""CAR""","""TB""",14,16,0,0.5,18.5625,22.75


In [68]:
# Gather home stats
home_stats = games_full.join( # Selects game_full as the left dataset
    games.select(['season', 'week', 'home_team', 'away_team']), # Selects these cols as the right dataset
    left_on=['season', 'week', 'team'],
    right_on=['season', 'week', 'home_team'] # Joins the sets together matching season to season, week to week, etc.
).rename({
    'team': 'home_team',
    'win_percentage': 'home_win_pct',
    'points_per_game': 'home_ppg',
    'allowed_points_per_game': 'home_opp_ppg'
}).select(['season', 'week', 'home_team', 'home_win_pct', 'home_ppg', 'home_opp_ppg'])


# Gather away stats
away_stats = games_full.join(
    games.select(['season', 'week', 'home_team', 'away_team']),
    left_on=['season', 'week', 'team'],
    right_on=['season', 'week', 'away_team']
).rename({
    'team': 'away_team',
    'win_percentage': 'away_win_pct',
    'points_per_game': 'away_ppg',
    'allowed_points_per_game': 'away_opp_ppg'
}).select(['season', 'week', 'away_team', 'away_win_pct', 'away_ppg', 'away_opp_ppg'])

gws = games.join(
    home_stats,
    on=['season', 'week', 'home_team'] # Joins the home stats where season, week, and home team match
).join(
    away_stats,
    on=['season', 'week', 'away_team'] # Same as above but for away stats
)

gws.tail()

season,game_type,week,away_team,away_score,home_team,home_score,result,total,overtime,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,home_team_win,home_win_pct,home_ppg,home_opp_ppg,away_win_pct,away_ppg,away_opp_ppg
i32,str,i32,str,i32,str,i32,i32,i32,i32,i32,i32,f64,i32,i32,f64,i32,i32,i8,f64,f64,f64,f64,f64,f64
2025,"""REG""",18,"""DAL""",17,"""NYG""",34,17,51,0,-166,140,-3.0,-115,-105,50.5,-115,-105,1,0.1875,21.6875,26.375,0.4375,28.375,29.8125
2025,"""REG""",18,"""WAS""",24,"""PHI""",17,-7,41,0,150,-180,3.0,-105,-115,38.5,-108,-112,0,0.6875,22.625,18.8125,0.25,20.75,27.125
2025,"""REG""",18,"""BAL""",24,"""PIT""",26,2,50,0,-225,185,-4.5,-105,-115,41.5,-110,-110,1,0.5625,23.1875,22.6875,0.5,25.0,23.25
2025,"""REG""",18,"""SEA""",13,"""SF""",3,-10,16,0,-148,124,-2.5,-120,100,47.5,-105,-115,0,0.75,27.125,22.375,0.8125,29.375,18.0625
2025,"""REG""",18,"""CAR""",14,"""TB""",16,2,30,0,130,-155,3.0,-115,-105,43.5,-118,-102,1,0.4375,22.75,24.8125,0.5,18.5625,22.75


In [69]:
# Training data (befor 2025 season)
gws_train = gws.filter(pl.col('season') < 2025)

# 2025 stats for prediction testing
gws_2025 = gws.filter(pl.col('season') == 2025)

# Write the data to csv files
gws_train.write_csv('nfl_features.csv')
gws_2025.write_csv('nfl_2025_stats.csv')

In [70]:
home = gws_2025.select([
    'season', 'week', 'home_team', 'home_win_pct', 'home_ppg', 'home_opp_ppg'
]).rename({
    'home_team': 'team',
    'home_win_pct': 'win_pct',
    'home_ppg': 'ppg',
    'home_opp_ppg': 'opp_ppg'
})

away = gws_2025.select([
    'season', 'week', 'away_team', 'away_win_pct', 'away_ppg', 'away_opp_ppg'
]).rename({
    'away_team': 'team',
    'away_win_pct': 'win_pct',
    'away_ppg': 'ppg',
    'away_opp_ppg': 'opp_ppg'
})

team_stats = pl.concat([home, away]).sort(['team', 'week']).group_by('team').last()

team_stats.write_csv('nfl_2025_final_stats.csv')

team_stats.head()


team,season,week,win_pct,ppg,opp_ppg
str,i32,i32,f64,f64,f64
"""CAR""",2025,18,0.5,18.5625,22.75
"""NYG""",2025,18,0.1875,21.6875,26.375
"""SF""",2025,18,0.75,27.125,22.375
"""NYJ""",2025,18,0.1875,18.25,29.25
"""LAC""",2025,18,0.6875,22.8125,20.0625
